In [1]:
%load_ext autoreload
%autoreload 2

import os
# Nạp biến môi trường (nếu file config.py của bạn chưa tự nạp)
from dotenv import load_dotenv
load_dotenv()

print("Môi trường test đã sẵn sàng!")

Môi trường test đã sẵn sàng!


In [2]:
# Import class từ file của bạn
from clients import FPTAIClient
from utils import JobMatcher
from schemas import TextRequest, MatchRequest
from config import settings

# Khởi tạo client
ai_client = FPTAIClient()
matcher = JobMatcher()

print(f"Đang dùng API tại: {settings.ENDPOINT}")

Đang dùng API tại: https://mkp-api.fptcloud.com


In [3]:
# Test 1: Refine CV
raw_text = "Mình tên Nam, làm kế toán 5 năm, biết Excel và Misa."
print("--- Đang gửi yêu cầu Refine ---")

try:
    refined_text = ai_client.chat_refine(raw_text)
    print("✅ KẾT QUẢ REFINE:")
    print(refined_text)
    for chunk in ai_client.chat_respond("Hãy recommend những công việc phù hợp nhất với tôi đi"):
        delta = chunk["choices"][0]["delta"].get("content", "")
        print(delta, end="", flush=True)
except Exception as e:
    print(f"❌ Lỗi: {e}")

--- Đang gửi yêu cầu Refine ---
✅ KẾT QUẢ REFINE:
Nam, 5 năm kinh nghiệm kế toán, thành thạo Excel và Misa.
<think>
Okay, let's see. The user is asking for job recommendations that suit them best. They mentioned they're an HR assistant. Hmm, I need to figure out what roles would be a good fit based on their current position.

First, I should consider the natural career progression from an HR Assistant. Common next steps might be HR Generalist or Specialist roles. Those positions build on the administrative and coordination skills they've already developed.

They might also be interested in areas like Talent Acquisition if they enjoy recruiting. Or maybe Training and Development if they like helping employees grow. Compensation and Benefits could be another path if they have experience with payroll or benefits administration.

I should also think about more specialized roles. Employee Relations Specialist comes to mind if they handle workplace issues. HRIS Analyst might be a good fit if

In [4]:
# Test 2: Embedding
print("--- Đang tạo Embedding ---")
vec_cv = ai_client.get_embedding(refined_text) # Dùng kết quả cell trên
vec_job = ai_client.get_embedding("Tuyển kế toán trưởng kinh nghiệm 5 năm, thành thạo Misa")

print(f"✅ Vector CV (10 số đầu): {vec_cv[:10]}")
print(f"Kích thước vector: {len(vec_cv)}")

# Test 3: Matching
score = matcher.compute_similarity(vec_cv, vec_job)
print(f"\n🎯 ĐỘ PHÙ HỢP: {score:.4f}")

--- Đang tạo Embedding ---
✅ Vector CV (10 số đầu): [0.012117696925997734, 0.006432757247239351, -0.005852817092090845, -0.028997007757425308, 0.011011231690645218, 0.030340027064085007, 0.054728034883737564, 0.04844026267528534, 0.008149685338139534, 0.04816555604338646]
Kích thước vector: 1024

🎯 ĐỘ PHÙ HỢP: 0.7894
